# 作业1：Data Types, Functions and Conditionals

- **学号**：20244081112
- **姓名**：杨俊堃
- **课程**：Python数据处理A

本 Notebook 按顺序完成四个部分：热身函数、欧几里得算法、自然常数 e 的近似、整数性质判断。
每个小节先给出实现，再给出测试与结论。

In [1]:
import os, uuid, hashlib, platform, getpass
# ===== 1. 填写学号 =====
STUDENT_ID = 20244081112
STUDENT_NAME = "杨俊堃"

# ===== 2. 打印 conda 环境 =====
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")

# ===== 3. 机器指纹（绑定当前电脑）=====
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]

# ===== 4. 生成唯一 ID =====
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]     {STUDENT_ID}")
print(f"[姓名]     {STUDENT_NAME}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境] exp01
[学号]     20244081112
[姓名]     杨俊堃
[机器指纹] b90e98a1cfa2cbc5
[RUN_ID]   20244081112-b90e98a1cfa2cbc5


---
## 一、热身：定义简单函数

三个函数都只涉及「定义 + 打印」，注意 `print_n` 用循环控制重复次数。

In [2]:
def say_hi():
    """不接受任何参数，打印问候语。"""
    print("Hello, world!")


def goat_pad(s):
    """接受一个字符串，打印其前后都加上 "goat" 的结果。"""
    print("goat" + s + "goat")


def print_n(s, n):
    """接受字符串 s 和整数 n，把 s 打印 n 次，每次单独一行。"""
    for _ in range(n):
        print(s)


print("--- say_hi() ---")
say_hi()

print("--- goat_pad('hello') ---")
goat_pad("hello")

print("--- print_n('goat', 3) ---")
print_n("goat", 3)

--- say_hi() ---
Hello, world!
--- goat_pad('hello') ---
goathellogoat
--- print_n('goat', 3) ---
goat
goat
goat


---
## 二、欧几里得算法

欧几里得算法基于一个事实：`gcd(m, n) == gcd(n, m % n)`，反复用除数替换被除数、用余数替换除数，
直到余数为 0，此时非零的除数就是最大公约数。

写成循环：

```
while n != 0:
    m, n = n, m % n
return m
```

In [3]:
def gcd(m, n):
    """返回 m 和 n 的最大公约数（欧几里得算法）。"""
    while n != 0:
        m, n = n, m % n
    return m

In [4]:
pairs = [(2018, 2019), (1200, 300), (5040, 60)]
for m, n in pairs:
    print(f"gcd({m}, {n}) = {gcd(m, n)}")

gcd(2018, 2019) = 1
gcd(1200, 300) = 300
gcd(5040, 60) = 60


### 3. 如果参数是负数会怎样？

在 Python 中 `%` 的结果符号跟随**除数**（第二个参数），因此对上面的实现：

| 调用 | 结果 | 说明 |
|---|---|---|
| `gcd(-2018, 2019)` | `1` | 除数为正，结果为正 |
| `gcd(2018, -2019)` | `-1` | 除数为负，结果为负 |
| `gcd(-1200, -300)` | `-300` | 除数为负，结果为负 |
| `gcd(1200, -300)` | `-300` | 除数为负，结果为负 |

**结论**：函数的返回值**绝对值**仍然正确（最大公约数只关心「哪个数能同时整除它们」），
但**符号会跟随第二个参数**。严格来说 `gcd` 按数学定义应返回非负整数，所以这种行为
「部分有意义」——它没有算错整除关系，但返回了一个负的「最大公约数」，不符合数学约定
（数学上 `gcd(-1200, -300) = 300`）。

一个更稳妥的写法是先取绝对值：

In [5]:
def gcd_safe(m, n):
    """先取绝对值，保证返回非负的最大公约数。"""
    m, n = abs(m), abs(n)
    while n != 0:
        m, n = n, m % n
    return m


print("--- 原始 gcd 遇到负数 ---")
for m, n in [(-2018, 2019), (2018, -2019), (-1200, -300), (1200, -300)]:
    print(f"gcd({m}, {n}) = {gcd(m, n)}")

print("--- 改进版 gcd_safe ---")
for m, n in [(-2018, 2019), (2018, -2019), (-1200, -300), (1200, -300)]:
    print(f"gcd_safe({m}, {n}) = {gcd_safe(m, n)}")

print("gcd_safe(-1200, -300) == gcd(1200, 300):", gcd_safe(-1200, -300) == gcd(1200, 300))

--- 原始 gcd 遇到负数 ---
gcd(-2018, 2019) = 1
gcd(2018, -2019) = -1
gcd(-1200, -300) = -300
gcd(1200, -300) = -300
--- 改进版 gcd_safe ---
gcd_safe(-2018, 2019) = 1
gcd_safe(2018, -2019) = 1
gcd_safe(-1200, -300) = 300
gcd_safe(1200, -300) = 300
gcd_safe(-1200, -300) == gcd(1200, 300): True


---
## 三、近似欧拉数 e

e 有两个等价的刻画：

1. 无穷级数：$e = \sum_{k=0}^{\infty} \frac{1}{k!} = 1 + 1 + \frac{1}{2} + \frac{1}{6} + \frac{1}{24} + \cdots$
2. 伯努利极限：$e = \lim_{x \to \infty} \left(1 + \frac{1}{x}\right)^x$

下面分别实现这两个式子，并给出「精度达到 $\epsilon$ 需要多少项」的版本。

In [6]:
import math


def euler_limit(n):
    """用极限式 (1 + 1/n)^n 近似 e。n 为正整数。"""
    return (1 + 1 / n) ** n


for n in [1, 2, 10, 100, 1000, 10000, 1000000]:
    print(f"n = {n:<9} (1 + 1/n)^n = {euler_limit(n):.10f}")

print(f"真实值 math.exp(1)        = {math.exp(1):.10f}")

n = 1         (1 + 1/n)^n = 2.0000000000
n = 2         (1 + 1/n)^n = 2.2500000000
n = 10        (1 + 1/n)^n = 2.5937424601
n = 100       (1 + 1/n)^n = 2.7048138294
n = 1000      (1 + 1/n)^n = 2.7169239322
n = 10000     (1 + 1/n)^n = 2.7181459268
n = 1000000   (1 + 1/n)^n = 2.7182804691
真实值 math.exp(1)        = 2.7182818285


In [7]:
def euler_infinite_sum(n):
    """返回级数前 n 项之和（n 为下标 0 到 n-1 的 n 项）。

    euler_infinite_sum(0) == 0，euler_infinite_sum(1) == 1。
    """
    total = 0.0
    factorial = 1          # 当前 k 的阶乘
    for k in range(n):     # k = 0, 1, ..., n-1
        if k > 0:
            factorial *= k
        total += 1.0 / factorial
    return total


for n in range(0, 8):
    print(f"euler_infinite_sum({n}) = {euler_infinite_sum(n)!r}")

print()
print("题目示例 euler_infinite_sum(4) ≈", euler_infinite_sum(4))
print("euler_infinite_sum(1) =", euler_infinite_sum(1))
print("euler_infinite_sum(0) =", euler_infinite_sum(0))

euler_infinite_sum(0) = 0.0
euler_infinite_sum(1) = 1.0
euler_infinite_sum(2) = 2.0
euler_infinite_sum(3) = 2.5
euler_infinite_sum(4) = 2.6666666666666665
euler_infinite_sum(5) = 2.708333333333333
euler_infinite_sum(6) = 2.7166666666666663
euler_infinite_sum(7) = 2.7180555555555554

题目示例 euler_infinite_sum(4) ≈ 2.6666666666666665
euler_infinite_sum(1) = 1.0
euler_infinite_sum(0) = 0.0


In [8]:
def euler_approx(epsilon):
    """累加级数，直到与真实 e 的误差小于 epsilon，返回该近似值。"""
    e_true = math.exp(1)
    n = 0            # 已累加的项数
    total = 0.0
    factorial = 1
    while abs(total - e_true) > epsilon:
        n += 1
        if n > 1:
            factorial *= (n - 1)
        total += 1.0 / factorial
    return total


for eps in [1e-1, 1e-3, 1e-6, 1e-10, 1e-15]:
    approx = euler_approx(eps)
    print(f"epsilon = {eps:<8g} 近似值 = {approx:.15f}  实际误差 = {abs(approx - math.exp(1)):.3e}")

epsilon = 0.1      近似值 = 2.666666666666667  实际误差 = 5.162e-02
epsilon = 0.001    近似值 = 2.718055555555555  实际误差 = 2.263e-04
epsilon = 1e-06    近似值 = 2.718281525573192  实际误差 = 3.029e-07
epsilon = 1e-10    近似值 = 2.718281828446759  实际误差 = 1.229e-11
epsilon = 1e-15    近似值 = 2.718281828459046  实际误差 = 4.441e-16


In [9]:
def print_euler_sum_table(n):
    """打印 k = 1..n 的 euler_infinite_sum(k)，每行一个。"""
    for k in range(1, n + 1):
        print(f"k = {k:<3} euler_infinite_sum(k) = {euler_infinite_sum(k)}")


print_euler_sum_table(10)

k = 1   euler_infinite_sum(k) = 1.0
k = 2   euler_infinite_sum(k) = 2.0
k = 3   euler_infinite_sum(k) = 2.5
k = 4   euler_infinite_sum(k) = 2.6666666666666665
k = 5   euler_infinite_sum(k) = 2.708333333333333
k = 6   euler_infinite_sum(k) = 2.7166666666666663
k = 7   euler_infinite_sum(k) = 2.7180555555555554
k = 8   euler_infinite_sum(k) = 2.7182539682539684
k = 9   euler_infinite_sum(k) = 2.71827876984127
k = 10  euler_infinite_sum(k) = 2.7182815255731922


### 几点说明

- `euler_infinite_sum` 每多取一项就多贡献 `1/k!`，收敛非常快，`n = 10` 已经有 7 位有效数字。
- `euler_limit` 收敛**很慢**：`n = 10^6` 时误差仍在 `1e-6` 量级，这正是 $\left(1+\frac1x\right)^x$ 的
  误差约为 $\frac{e}{2x}$ 的体现。
- `euler_approx` 采用「先判断、后累加」的 while 循环，因此当 `epsilon ≥ e`（例如 `epsilon = 3`）
  时循环一次都不执行，直接返回 `0.0`——这是输入超出合理范围时的边界情况。

---
## 四、测试整数属性

一个正整数 $n$ 是 2 的幂，当且仅当它不断除以 2 后最终得到 1。
推广到任意底数 $b$：$n$ 是 $b$ 的幂，当且仅当不断除以 $b$ 后最终得到 1。
整个实现只用除法和取模。

In [10]:
def is_power_of_2(n):
    """判断正整数 n 是否为 2 的幂（只用 // 和 %）。"""
    while n % 2 == 0:
        n = n // 2
    return n == 1


for n in [1, 2, 3, 4, 5, 8, 16, 64, 100, 1024, 2018]:
    print(f"is_power_of_2({n:<5}) = {is_power_of_2(n)}")

is_power_of_2(1    ) = True
is_power_of_2(2    ) = True
is_power_of_2(3    ) = False
is_power_of_2(4    ) = True
is_power_of_2(5    ) = False
is_power_of_2(8    ) = True
is_power_of_2(16   ) = True
is_power_of_2(64   ) = True
is_power_of_2(100  ) = False
is_power_of_2(1024 ) = True
is_power_of_2(2018 ) = False


In [11]:
def is_power(b, n):
    """判断正整数 n 是否为底数 b 的幂。b = n 时也返回 True（n^1）。"""
    if b == 1:          # 1 的任何次幂都是 1，需单独处理，否则会死循环
        return n == 1
    while n % b == 0:
        n = n // b
    return n == 1


print("--- 底数为 3 ---")
for n in [1, 3, 9, 27, 81, 12, 30]:
    print(f"is_power(3, {n:<4}) = {is_power(3, n)}")

print("--- 其他底数 ---")
for b, n in [(2, 64), (2, 63), (5, 125), (5, 124), (7, 7), (10, 1000), (1, 1), (1, 2)]:
    print(f"is_power({b}, {n:<5}) = {is_power(b, n)}")

print()
print("一致性检查：is_power(2, n) 与 is_power_of_2(n) 对 1..1024 是否完全一致 ->",
      all(is_power(2, n) == is_power_of_2(n) for n in range(1, 1025)))

--- 底数为 3 ---
is_power(3, 1   ) = True
is_power(3, 3   ) = True
is_power(3, 9   ) = True
is_power(3, 27  ) = True
is_power(3, 81  ) = True
is_power(3, 12  ) = False
is_power(3, 30  ) = False
--- 其他底数 ---
is_power(2, 64   ) = True
is_power(2, 63   ) = False
is_power(5, 125  ) = True
is_power(5, 124  ) = False
is_power(7, 7    ) = True
is_power(10, 1000 ) = True
is_power(1, 1    ) = True
is_power(1, 2    ) = False

一致性检查：is_power(2, n) 与 is_power_of_2(n) 对 1..1024 是否完全一致 -> True


### 小结

- `is_power_of_2` 是 `is_power` 在 `b = 2` 时的特例，两者结果完全一致（上面已用 1~1024 全量校验）。
- 两个函数都只用 `%`（判断能否整除）和 `//`（缩小规模），没有用到 `math.sqrt`。
- `b = 1` 是必须单独处理的边界：`1 % 1 == 0` 恒成立，会陷入死循环。
- 时间复杂度为 $O(\log_b n)$。